# 03 — Confidence Calibration via Temperature Scaling

This notebook applies post-hoc temperature scaling (Guo et al., 2017) to the trained EfficientNet-B3 detector from notebook 02.

**What temperature scaling does:** Modern neural networks tend to be overconfident — a model reporting 95% confidence may only be correct 80% of the time. Temperature scaling learns a single parameter *T* that softens the model's output distribution:

```
calibrated_probs = softmax(logits / T)
```

When *T* > 1, predictions become less extreme (closer to uniform), reducing overconfidence.

**This notebook:**
1. Loads the trained checkpoint and validation data
2. Collects raw logits on the validation set
3. Measures calibration **before** temperature scaling (ECE + reliability diagram)
4. Learns the optimal temperature *T* on the validation set
5. Measures calibration **after** temperature scaling (ECE + reliability diagram)
6. Produces a side-by-side comparison plot
7. Saves the results to `baseline_results.json`

**Prerequisites:** Run `02_baseline_training.ipynb` first to produce a trained checkpoint.

In [1]:
import os
from google.colab import drive
drive.mount("/content/drive")

REPO_DIR = "/content/ai-image-detection"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/krishi-shah/ai-image-detection.git {REPO_DIR}
os.chdir(REPO_DIR)

Mounted at /content/drive
Cloning into '/content/ai-image-detection'...
remote: Enumerating objects: 77, done.
remote: Counting objects: 100% (77/77), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 77 (delta 34), reused 77 (delta 34), pack-reused 0 (from 0)
Receiving objects: 100% (77/77), 3.00 MiB | 6.42 MiB/s, done.
Resolving deltas: 100% (34/34), done.


In [2]:
!pip install -q -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 61.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 90.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 91.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 86.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 15.1 MB/s eta 0:0

In [3]:
import os, zipfile, glob
DATA_DIR = "/content/ai-image-detection/data/raw/cifake"
DRIVE_ROOT = "/content/drive/MyDrive/ai-image-detection"
os.makedirs(DATA_DIR, exist_ok=True)
if not os.path.exists(os.path.join(DATA_DIR, "train")):
    zip_path = glob.glob(os.path.join(DRIVE_ROOT, "*.zip"))[0]
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(DATA_DIR)
    print("Extracted.")

Extracted.


In [4]:
# Cell 1 — Imports, paths, device, load checkpoint
import os, sys, json
import numpy as np
import torch
import torch.nn as nn

REPO_DIR = "/content/ai-image-detection"
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

DRIVE_ROOT = "/content/drive/MyDrive/ai-image-detection"
DATA_DIR = os.path.join(REPO_DIR, "data", "raw", "cifake")
CHECKPOINT_DIR = os.path.join(DRIVE_ROOT, "checkpoints")
PLOTS_DIR = os.path.join(REPO_DIR, "outputs", "plots")
RESULTS_DIR = os.path.join(REPO_DIR, "outputs", "results")
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

from src.model.detector import build_detector, load_checkpoint
from src.utils.data_loader import get_cifake_loaders

model = build_detector(pretrained=False).to(device)
checkpoint_path = os.path.join(CHECKPOINT_DIR, "best_detector.pth")
epoch_loaded, ckpt_metrics = load_checkpoint(model, checkpoint_path)
model.to(device)
model.eval()

print(f"Loaded checkpoint from epoch {epoch_loaded}")
print(f"  val_acc: {ckpt_metrics['val_acc']:.4f}")

_, val_loader, test_loader = get_cifake_loaders(DATA_DIR, batch_size=32, num_workers=2)

Device: cuda
Loaded checkpoint from epoch 2
  val_acc: 0.9694


In [5]:
# Cell 2 — Collect raw logits on the validation set
from src.evaluation.calibration import collect_logits

val_logits, val_labels = collect_logits(model, val_loader, device)

print(f"Collected logits: {val_logits.shape}")
print(f"Labels:           {val_labels.shape}")
print(f"Class distribution: {dict(zip(*np.unique(val_labels.numpy(), return_counts=True)))}")

Collected logits: torch.Size([20000, 2])
Labels:           torch.Size([20000])
Class distribution: {np.int64(0): np.int64(10038), np.int64(1): np.int64(9962)}


## Pre-Calibration Analysis

Before applying temperature scaling, we measure how well the model's raw confidence scores match its actual accuracy. A perfectly calibrated model sits on the diagonal of the reliability diagram.

In [6]:
# Cell 3 — Pre-calibration: ECE and reliability diagram
from src.evaluation.metrics import compute_ece, plot_reliability_diagram

probs_before = torch.softmax(val_logits, dim=1).numpy()
ece_before = compute_ece(probs_before, val_labels.numpy())

print(f"ECE (before temperature scaling): {ece_before:.4f}")

plot_reliability_diagram(
    probs_before,
    val_labels.numpy(),
    os.path.join(PLOTS_DIR, "reliability_pre_calibration.png"),
)
print(f"Saved: {PLOTS_DIR}/reliability_pre_calibration.png")

ECE (before temperature scaling): 0.0031
Saved: /content/ai-image-detection/outputs/plots/reliability_pre_calibration.png


## Learn Optimal Temperature

We optimise a single scalar *T* on the validation set using L-BFGS to minimise the negative log-likelihood. This is the method from Guo et al. (2017) — despite having only one parameter, it matches or outperforms more complex calibration methods.

In [7]:
# Cell 4 — Learn temperature on validation set
from src.evaluation.calibration import find_temperature

learned_temperature = find_temperature(model, val_loader, device)

print(f"\nLearned temperature: {learned_temperature:.4f}")
if learned_temperature > 1.0:
    print("T > 1 indicates the model was overconfident (as expected for modern DNNs).")
else:
    print("T <= 1 indicates the model was underconfident.")

Temperature: 1.2189
ECE before:  0.0031
ECE after:   0.0116

Learned temperature: 1.2189
T > 1 indicates the model was overconfident (as expected for modern DNNs).


## Post-Calibration Analysis

Now we apply the learned temperature to the same validation logits and re-measure ECE. If calibration worked, the bars in the reliability diagram should move closer to the diagonal and ECE should decrease.

In [8]:
# Cell 5 — Post-calibration: ECE and reliability diagram
from src.evaluation.calibration import apply_temperature

probs_after = apply_temperature(val_logits.numpy(), learned_temperature)
ece_after = compute_ece(probs_after, val_labels.numpy())

print(f"ECE (after temperature scaling):  {ece_after:.4f}")
print(f"ECE reduction:                    {ece_before - ece_after:.4f}")

plot_reliability_diagram(
    probs_after,
    val_labels.numpy(),
    os.path.join(PLOTS_DIR, "reliability_post_calibration.png"),
)
print(f"Saved: {PLOTS_DIR}/reliability_post_calibration.png")

ECE (after temperature scaling):  0.0116
ECE reduction:                    -0.0084
Saved: /content/ai-image-detection/outputs/plots/reliability_post_calibration.png


## Side-by-Side Comparison

A combined figure showing the reliability diagram before and after calibration. The dotted diagonal represents perfect calibration.

In [9]:
# Cell 6 — Side-by-side reliability diagrams
import matplotlib.pyplot as plt

n_bins = 10
bin_boundaries = np.linspace(0.0, 1.0, n_bins + 1)
bin_centres = [(bin_boundaries[i] + bin_boundaries[i + 1]) / 2 for i in range(n_bins)]
width = 1.0 / n_bins


def _bin_accuracies(probs, labels):
    confidences = np.max(probs, axis=1)
    predictions = np.argmax(probs, axis=1)
    accuracies = (predictions == labels).astype(float)
    accs = []
    for i in range(n_bins):
        mask = (confidences > bin_boundaries[i]) & (confidences <= bin_boundaries[i + 1])
        accs.append(float(accuracies[mask].mean()) if mask.sum() > 0 else 0.0)
    return accs


labels_np = val_labels.numpy()
accs_before = _bin_accuracies(probs_before, labels_np)
accs_after = _bin_accuracies(probs_after, labels_np)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.bar(bin_centres, accs_before, width=width * 0.9, alpha=0.7, label="Accuracy")
ax1.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
ax1.set_xlabel("Confidence")
ax1.set_ylabel("Accuracy")
ax1.set_title(f"Before (ECE = {ece_before:.4f})")
ax1.legend(loc="upper left")
ax1.set_xlim(0, 1)
ax1.set_ylim(0, 1)

ax2.bar(bin_centres, accs_after, width=width * 0.9, alpha=0.7, color="green", label="Accuracy")
ax2.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
ax2.set_xlabel("Confidence")
ax2.set_ylabel("Accuracy")
ax2.set_title(f"After  (ECE = {ece_after:.4f}, T = {learned_temperature:.2f})")
ax2.legend(loc="upper left")
ax2.set_xlim(0, 1)
ax2.set_ylim(0, 1)

fig.suptitle("Reliability Diagrams: Before vs After Temperature Scaling", fontsize=14)
fig.tight_layout()
fig.savefig(os.path.join(PLOTS_DIR, "calibration_comparison.png"), dpi=150)
plt.show()
print(f"Saved: {PLOTS_DIR}/calibration_comparison.png")

Saved: /content/ai-image-detection/outputs/plots/calibration_comparison.png


## Test Set Evaluation + Save Results

Finally, we evaluate the calibrated model on the **test set** and save all metrics to a JSON file. This file will be used in Phase 2 as the baseline for comparison against generalisation results on other generators.

In [10]:
# Cell 7 — Test set evaluation with calibration + save results
from src.evaluation.metrics import compute_accuracy, compute_auc
from tqdm.auto import tqdm

test_logits, test_labels = collect_logits(model, test_loader, device)

test_probs_before = torch.softmax(test_logits, dim=1).numpy()
test_probs_after = apply_temperature(test_logits.numpy(), learned_temperature)

test_preds = np.argmax(test_probs_before, axis=1)
test_labels_np = test_labels.numpy()

test_acc = compute_accuracy(test_preds, test_labels_np)
test_auc = compute_auc(test_probs_before, test_labels_np)
test_ece_before = compute_ece(test_probs_before, test_labels_np)
test_ece_after = compute_ece(test_probs_after, test_labels_np)

print(f"{'='*45}")
print(f"  Test Results (CIFAKE) — Calibrated")
print(f"{'='*45}")
print(f"  Accuracy:              {test_acc:.4f}")
print(f"  AUC-ROC:               {test_auc:.4f}")
print(f"  ECE (before scaling):  {test_ece_before:.4f}")
print(f"  ECE (after scaling):   {test_ece_after:.4f}")
print(f"  Temperature:           {learned_temperature:.4f}")
print(f"{'='*45}")

# Save structured results for Phase 2 comparison
results = {
    "dataset": "CIFAKE",
    "model": "EfficientNet-B3",
    "checkpoint_epoch": int(epoch_loaded),
    "test_accuracy": round(test_acc, 4),
    "test_auc": round(test_auc, 4),
    "ece_before_calibration": round(test_ece_before, 4),
    "ece_after_calibration": round(test_ece_after, 4),
    "temperature": round(learned_temperature, 4),
    "checkpoint_path": "outputs/checkpoints/best_detector.pth",
}

results_path = os.path.join(RESULTS_DIR, "baseline_results.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"\nResults saved to {results_path}")

  Test Results (CIFAKE) — Calibrated
  Accuracy:              0.9696
  AUC-ROC:               0.9971
  ECE (before scaling):  0.0026
  ECE (after scaling):   0.0113
  Temperature:           1.2189

Results saved to /content/ai-image-detection/outputs/results/baseline_results.json


In [11]:
import shutil
shutil.copytree(
    "/content/ai-image-detection/outputs",
    "/content/drive/MyDrive/ai-image-detection/outputs",
    dirs_exist_ok=True
)
print("Outputs saved to Drive!")

Outputs saved to Drive!


## Threshold Sensitivity Analysis

The default decision threshold is 0.5 — whichever class has the higher probability wins. But different thresholds trade off precision against recall. A lower threshold (e.g. 0.4) catches more fakes but produces more false positives; a higher threshold (e.g. 0.6) only flags high-confidence fakes but misses more.

Below we sweep across thresholds from 0.40 to 0.70 and compute **Accuracy, Precision, Recall, and F1** at each one, treating FAKE (class 0) as the positive class.

In [12]:
# Cell 8 — Threshold sensitivity analysis
import json
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]
fake_probs = test_probs_before[:, 0]  # P(FAKE) for each sample

records = []
for t in thresholds:
    preds = (fake_probs >= t).astype(int)
    preds_class = 1 - preds  # flip: 1 where FAKE, but labels use 0=FAKE
    preds_class = np.where(fake_probs >= t, 0, 1)

    acc  = compute_accuracy(preds_class, test_labels_np)
    prec = precision_score(test_labels_np, preds_class, pos_label=0)
    rec  = recall_score(test_labels_np, preds_class, pos_label=0)
    f1   = f1_score(test_labels_np, preds_class, pos_label=0)

    records.append({"threshold": t, "accuracy": acc, "precision": prec, "recall": rec, "f1": f1})

# Print table
print(f"{'Threshold':>10}  {'Accuracy':>9}  {'Precision':>10}  {'Recall':>8}  {'F1':>8}")
print("-" * 55)
for r in records:
    marker = "  <-- default" if r["threshold"] == 0.50 else ""
    print(f"{r['threshold']:>10.2f}  {r['accuracy']:>9.4f}  {r['precision']:>10.4f}  {r['recall']:>8.4f}  {r['f1']:>8.4f}{marker}")

# Extract arrays for plotting
t_vals   = [r["threshold"] for r in records]
acc_vals  = [r["accuracy"] for r in records]
prec_vals = [r["precision"] for r in records]
rec_vals  = [r["recall"] for r in records]
f1_vals   = [r["f1"] for r in records]

# --- Plot 1: Accuracy vs Threshold ---
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(t_vals, acc_vals, "o-", linewidth=2)
ax.axvline(0.5, color="gray", linestyle="--", alpha=0.7, label="Default (0.5)")
ax.set_xlabel("FAKE-class Threshold")
ax.set_ylabel("Accuracy")
ax.set_title("Accuracy vs Decision Threshold")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(PLOTS_DIR, "threshold_accuracy.png"), dpi=150)
plt.show()
print(f"Saved: {PLOTS_DIR}/threshold_accuracy.png")

# --- Plot 2: Precision & Recall vs Threshold ---
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(t_vals, prec_vals, "s-", linewidth=2, label="Precision")
ax.plot(t_vals, rec_vals, "^-", linewidth=2, label="Recall")
ax.axvline(0.5, color="gray", linestyle="--", alpha=0.7, label="Default (0.5)")
ax.set_xlabel("FAKE-class Threshold")
ax.set_ylabel("Score")
ax.set_title("Precision & Recall vs Decision Threshold")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(PLOTS_DIR, "threshold_precision_recall.png"), dpi=150)
plt.show()
print(f"Saved: {PLOTS_DIR}/threshold_precision_recall.png")

# --- Plot 3: F1 vs Threshold ---
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(t_vals, f1_vals, "D-", linewidth=2, color="green")
ax.axvline(0.5, color="gray", linestyle="--", alpha=0.7, label="Default (0.5)")
ax.set_xlabel("FAKE-class Threshold")
ax.set_ylabel("F1 Score")
ax.set_title("F1 Score vs Decision Threshold")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(PLOTS_DIR, "threshold_f1.png"), dpi=150)
plt.show()
print(f"Saved: {PLOTS_DIR}/threshold_f1.png")

# --- Plot 4: All metrics combined ---
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(t_vals, acc_vals, "o-", linewidth=2, label="Accuracy")
ax.plot(t_vals, prec_vals, "s-", linewidth=2, label="Precision")
ax.plot(t_vals, rec_vals, "^-", linewidth=2, label="Recall")
ax.plot(t_vals, f1_vals, "D-", linewidth=2, label="F1")
ax.axvline(0.5, color="gray", linestyle="--", alpha=0.7, label="Default (0.5)")
ax.set_xlabel("FAKE-class Threshold")
ax.set_ylabel("Score")
ax.set_title("All Metrics vs Decision Threshold (CIFAKE Test Set)")
ax.legend(loc="lower left")
ax.grid(True, alpha=0.3)
ax.set_ylim(0.85, 1.01)
fig.tight_layout()
fig.savefig(os.path.join(PLOTS_DIR, "threshold_all_metrics.png"), dpi=150)
plt.show()
print(f"Saved: {PLOTS_DIR}/threshold_all_metrics.png")

# Save results to JSON
threshold_results_path = os.path.join(RESULTS_DIR, "threshold_analysis.json")
with open(threshold_results_path, "w") as f:
    json.dump(records, f, indent=2)
print(f"\nThreshold analysis saved to {threshold_results_path}")

 Threshold   Accuracy   Precision    Recall        F1
-------------------------------------------------------
      0.40     0.9722      0.9807    0.9635    0.9720
      0.45     0.9716      0.9839    0.9589    0.9712
      0.50     0.9696      0.9858    0.9529    0.9691  <-- default
      0.55     0.9665      0.9871    0.9454    0.9658
      0.60     0.9646      0.9894    0.9393    0.9637
      0.65     0.9605      0.9906    0.9297    0.9592
      0.70     0.9563      0.9922    0.9197    0.9546
Saved: /content/ai-image-detection/outputs/plots/threshold_accuracy.png
Saved: /content/ai-image-detection/outputs/plots/threshold_precision_recall.png
Saved: /content/ai-image-detection/outputs/plots/threshold_f1.png
Saved: /content/ai-image-detection/outputs/plots/threshold_all_metrics.png

Threshold analysis saved to /content/ai-image-detection/outputs/results/threshold_analysis.json


In [13]:
# Cell 9 — Save all outputs (including threshold analysis) to Google Drive
import shutil
shutil.copytree(
    os.path.join(REPO_DIR, "outputs"),
    "/content/drive/MyDrive/ai-image-detection/outputs",
    dirs_exist_ok=True
)
print("All outputs (including threshold analysis) saved to Drive!")

All outputs (including threshold analysis) saved to Drive!
